In [ ]:
import numpy as np
import pandas as pd
import re
from pandarallel import pandarallel
import yaml

pandarallel.initialize(progress_bar=True)

In [ ]:
import os
os.chdir('../../')

In [ ]:
# Load configuration settings from a YAML file.
# The configuration file typically contains file paths and other parameters.
dataset_config = yaml.safe_load(open("config/dataset.yml"))
dataset_config

## Import papers in the CNKI with affiliation

In [ ]:
author_list = pd.read_csv(dataset_config['path_cnki'] +'author_list.txt', encoding='utf-8', delimiter='|').dropna()
author_list

## Clean the data

In [ ]:
def contains_chinese(text):
    """
    Check if the given string contains any Chinese characters.

    Parameters:
    - text: string to check

    Returns:
    - True if the string contains any Chinese characters, False otherwise.
    """
    # Regex pattern to match Chinese characters (CJK Ideographs)
    chinese_char_pattern = re.compile(r'[\u4e00-\u9fff]')

    # If any Chinese character is found, return True
    return bool(chinese_char_pattern.search(text))

author_list['contains_Chinese'] = author_list['author_add'].apply(contains_chinese)
author_list = author_list[author_list['contains_Chinese']][['paperid', 'author_name', 'author_add']]
author_list

In [ ]:
def keep_before_space(text):
    """
    Keep only the part of the string before the first space.

    Parameters:
    - text: the input string

    Returns:
    - The part of the string before the first space, or the entire string if no space exists.
    """
    # Split the string at the first space and return the first part
    return text.split(' ', 1)[0] if text else text

author_list['author_add_cleaned'] = author_list['author_add'].apply(keep_before_space)
author_list

In [ ]:
def split_author_add(value):
    if pd.isna(value):  # Check if the value is NaN
        return value
    return [item.strip().split('!')[0] for item in value.split(';')]

author_list['author_add_cleaned'] = author_list['author_add_cleaned'].apply(split_author_add)
author_list

In [ ]:
author_list['author_add_cleaned'] = author_list['author_add_cleaned'].apply(lambda x: [item for item in x if contains_chinese(item)] if isinstance(x, list) else [])
author_list

In [ ]:
# Function to check if the string matches any of the unwanted patterns
def contains_unwanted_pattern(item):
    # Regular expression for 1 to 3 Chinese characters followed by exactly 6 digits
    pattern1 = r'[\u4e00-\u9fff]{1,4}\d{6}$'
    # Regular expression for 1 or 2 Chinese characters, without checking digits
    pattern2 = r'^[\u4e00-\u9fff]{1,2}$'
    # Regular expression for exactly 6 digits followed by 1 to 3 Chinese characters
    pattern3 = r'\d{6}[\u4e00-\u9fff]{1,4}$'

    # Return True if any of the unwanted patterns are matched
    return (bool(re.match(pattern1, item)) or
            bool(re.match(pattern2, item)) or
            bool(re.match(pattern3, item)))

# Function to remove unwanted content like parentheses
def clean_item(item):
    # Remove contents inside parentheses
    item = re.sub(r'[\(\（][^()\（\）]*[\)\）]', '', item)
    # Remove any extra spaces
    item = re.sub(r'\s+', '', item)  
    return item

# Applying the function to filter out the undesired entries
author_list['author_add_cleaned'] = author_list['author_add_cleaned'].apply(
    lambda x: [clean_item(item) for item in x if not contains_unwanted_pattern(item)] if isinstance(x, list) else []
)

author_list

In [ ]:
author_list[author_list['paperid'] == '17490-16']

## Classification: (1) firm, (2) university or research institue, and (3) government or others

In [ ]:
def categorize_author_add(author_add_list):
    config_categories = {
                        'f': (['公司', '集团','株式会社','事务所','企业','证券','有限责任','(株)'], 
                               ['厂','行']),
                        'u': (['大学', '学院', '学校', '研究', '实验室', '教学基地', '实验站', '学会', '小学','中学','科研所'], 
                               ['所', '系', '院', '中心','中','大','高','校','专']),
                        'g': (['医院', '海关', '委员会','党校','医学中心','卫生中心','政府','世界卫生组织'], 
                               ['局','站','司','街道','园','队','村','屯','庄','旗','团','会','协','厅','连','师','场','医','管理处','组织','协会','妇联','段','圃','组','办','部','委','社','署','馆','区','科','门诊','卫生室','库','台','区','室','处','项目','大队','病房','基地','协作组','军医','icu','ICU','产房','养区','病区'])
                        }

    # If author_add_list is not a list, return 'n' as a list
    if type(author_add_list) is not list:
        return ['n']

    result = []

    # Check for keywords related to companies (category 'f')
    for item in author_add_list:
        flag_found = False
        for cat_key in config_categories.keys():
            if any(keyword in item for keyword in config_categories[cat_key][0]):
                result.append(cat_key)
                flag_found = True
                break
            
            elif any(item.endswith(keyword) for keyword in config_categories[cat_key][1]):
                result.append(cat_key)
                flag_found = True
                break

        # If no categories matched, add 'o' to result list
        if not flag_found:
            result.append('o')

    return result

In [ ]:
author_list['category'] = author_list['author_add_cleaned'].parallel_apply(categorize_author_add)
author_list

## Extract papers published by firms

In [ ]:
def contains_f(L):
    if 'f' in L:
        return [idx for idx in range(len(L)) if L[idx] == 'f']
    else:
        return False
    
author_list['contain_firms'] = author_list['category'].apply(contains_f)
author_list


In [ ]:
list_with_firms = author_list[author_list['contain_firms'] != False]
list_with_firms

In [ ]:
# Export papers published by firms for further analysis
list_with_firms[['paperid', 'author_add_cleaned']].to_parquet(dataset_config['path_processed'] + 'CNKI/CNKI_paperid_CNfirm.parquet')

## Match with Entity List

In [ ]:
def locate_entity(L):

    entities_2018 = [
        "中国电子科技集团公司第十三研究所", "中国电子科技集团公司第三十八研究所", "中国电子科技集团公司第五十五研究所", 
        "中国华腾工业有限公司", "河北远东通信", "成都佳士科技有限公司", "成都海威华芯科技有限公司", 
        "福建省晋华集成电路有限公司"
    ]

    entities_2019 = [
        "成都海光", "华为", "曙光信息产业", "无锡江南计算技术研究所", 
        "中国广核", "中广核研究院", "苏州热工研究院", "浙江大华技术", "海康威视", "科大讯飞", 
        "北京旷视科技有限公司", "商汤科技", "厦门市美亚柏科信息", "上海依图络科技", "北京亿欣科技"
    ]

    entities_2020 = [
    "武汉市伊尔森科技", "北京中云融信科技", "北京锦程环宇科贸", "中国九原", "哈尔滨创越科技", "哈尔滨蕴力达", "奇虎360", "上海诺瓦仪器", 
        "四川鼎澄物资", "四川新天元科技", "四川图斯克进出口", "砺剑天眼科技", "快急送物流", "阿克苏华孚色纺", "云从科技集团", "烽火科技集团", 
        "南京峰火星空通信发展", "北京东方网力", "深圳市深网视界", "深圳云天励飞", "易思维", "华大基因", "昌吉溢达纺织", "合肥宝龙达", "合肥美菱", 
        "新疆和田浩林发饰品", "和田泰达服饰", "今创集团", "南京新一棉纺织印染", "南昌欧菲光", "碳元科技", "中交疏装", 
        "中交天津航道局", "中交上海航道局", "中交广州航道局", "中交第二航务工程局", "北京市环佳", "常州国光", "中国电子科技集团公司第七研究所", 
        "北京崇新八达科技开发", "广州广有通信设备", "广州海格通信集团", "桂林长海发展", "湖北广兴通信", "陕西长岭电子科技", "上海凯波水下工程", 
        "北京特立信电子技术", "天津广播器材", "天津764", "武汉迈力特通信", "无锡中德美联生物技术", "中国交通建设", "中国科学仪器和材料公司", 
        "重庆川东船舶重工", "中船黄埔文冲船舶", "大疆", "广新海事重工", "广州市泰诚船舶工业", "江苏恒翔智产信息技术", "深圳光启集团", 
        "南京资产管理", "宁波半导体国际公司", "诺思微系统", "中芯国际", "中芯南方集成电路", "中芯长电半导体", "中芯国际控股", "中芯北方集成电路", 
        "中芯半导体", "同方威视技术", "中国航空发动机", "中国航发航空科技", "中国航发动力", "中国航发燃气轮机", "中国航发商用航空发动机", 
        "中国航发哈尔滨东安发动机", "中国航发沈阳黎明航空科技", "中国航发湖南南方宇航工业", "安徽应流航源动力科技", "中国航空工业", "中航飞机", 
        "成都飞机工业", "中航通飞华南飞机工业", "浙工中航通飞研究院", "中航国际", "北京百慕航材", "北京安达泰克",
        "北京力威尔航空精密机械", "北京光明远大电子", "西安航天发动机", "成都航利航空科技", "中国航空工业标准件制造",
        "中船西工造船", "上海屹领电子科技", "合肥市富华精密机械制造", "广州航新航空科技", "贵州航宇科技发展",
        "贵州黎阳国际制造", "中航直升机", "杭州轴承试验研究中心", "哈尔滨通用飞机工业", "河南航天精工制造",
        "湖南南方通用航空发动机", "重庆奥普泰通信技术", "江苏美龙航空部件", "湖北航宇嘉泰飞机设备", "金城集团进出口",
        "美迪希实验仪器", "陕西航空电气", "陕西飞机工业", "上海航天设备制造", "上海飞机制造",
        "上海天朗电子技术", "沈阳仪表科学研究院", "沈阳飞机工业", "内阳西子航空产业", "四川航特航空设备",
        "诚泰航材设备", "江苏苏美达仪器设备", "苏州市意可机电", "无锡航亚科技", "无锡派克新材料科技",
        "无锡透平叶片", "西安飞机工业航电进出口", "西飞科技工贸", "中国航发西安动力控制", "西安飞机工业",
        "西安西航集团莱特航空制造技术", "西安西罗航空部件", "宜宾三江机械", "浙江完美新材料科技"
    ]

    entities_2021 = [
        "中国海洋石油", "北京天骄航空产业投资", "信维微电子", "天津飞腾信息技术", "合盛硅业", "新疆大全新能源",
        "新疆东方希望有色金属", "新疆协鑫新能源材料科技", "北京东土军悦科技", "北京燕京电子", "北京格灵深瞳信息技术",
        "北京海力联合科技", "北京中电兴发料技", "成都西物信安智能系统", "杭州华澜微电子", "上海金卓科技",
        "北京东土科技", "立昂技术", "深圳市科葩信息技术", "深圳市华安泰智能科技", "苏州科达科技",
        "同方锐安科技", "乌鲁木齐天耀伟业信息技术服务", "武汉锐科光纤激光技术", "新最北斗同创信息科技", "新疆联海创智信息科技",
        "新疆熙菱信息技术", "新疆汤立科技", "深圳嘉兆科技", "杭州中科微电子", "湖南国科微电子",
        "新华三半导体技术", "保利亚太", "科大国盾量子技术", "陕西智恩机电科技", "上海国盾量子信息技术",
        "西安航天华讯科技", "苏州云芯微电子科技", "航天展光", "长沙景嘉微电子", "康泰尔科技",
        "福建火炬电子科技", "杭州海康微影传感科技", "华海通信国际", "香港昌华电子科技", "华视界电子",
        "超系统联盟", "内蒙古第一机械集团", "江苏亨通海洋光网系统", "江苏亨通光电", "陕西亚成微电子",
        "上海爱信诺航芯电子科技", "上海做世控制科技", "深圳市瑞芬科技", "鸣洋电气", "中天科技海缆"
    ]

    entities_2022 = [
        "柯亲柯特电子", "香港世捷达物流", "信诺电子", "维科电子", "旺年华电子",
        "北京海兰信数据科技", "中船电子科技", "三沙海兰信数据科技", "三亚海兰寰宇海洋信息科技", "安徽寒武纪信息科技",
        "安洲国际集团", "北京华天海峰科技", "北京机械工业自动化研究所", "北京合众思壮科技", "北京银景科技",
        "寒武纪", "电科云科技", "中电来斯信息系统", "广东琴智科技研究院", "合肥兆芯电子",
        "南京艾溪信息科技", "南京来斯网信技术研究院", "南京莱斯电子设备", "南京莱斯信息技术", "深圳市鹏芯微集成电路制造",
        "上海寒武纪信息科技", "上海微电子装备", "上海索为信息科技", "苏州寒武纪信息科技", "溧阳二十八所系统装备",
        "天津天地伟业科技", "雄安寒武纪科技", "长江存储科技", "北京中科新联科技", "长江存储技术"
    ]

    entities_2023 = [
        "北京南江空天科技", "东莞凌空遥感科技", "铱格斯曼航空科技", "广州天海翔航空科技", "山西铱格斯曼航空科技",
        "AOOK科技", "北京天泰科技发展", "北京昀泽科技", "德宇航技术", "长沙无仪空间科技研究院",
        "北京第四范式智能技术", "全翔国际货运", "爱速实业", "保定巨人进出国", "保定市世资通企业服务",
        "北京正远创势各询", "深圳华大法医科技", "保定开拓精密仪器制造", "浪潮集团", "龙芯中科技术",
        "南京哥伦巴机械设备", "南京久鼎环境科技", "上海瞬渺光电技术", "苏州盛科通信设份", "苏州盛科科技",
        "洛浦海视鼎鑫电子技术", "墨玉海视电子技术", "皮山海视永安电子技水", "乌鲁木齐海视新安电子技术", "于田海视美阗电予技术",
        "立维科技有限公司", "亿商网络", "永利电子元器件", "中航国际仿真科技服务", "中国航空技术北京",
        "北京亿维讯同创科技", "北京罗罗科技发展", "北京瑞源文德科技", "北京天圣华信息技术", "北京天圣华科技",
        "贝尔特顾问", "常州优泰克复合材料", "成都搏洋腾空科技", "中国天利航空", "益强国际贸易",
        "先丰服务", "大众科技", "新信实业投资", "欧唐", "安世亚太科技",
        "深圳千普科技", "上海航天科学技术开发", "上海轻风科技", "上海轻风科技江苏", "上海超算科技",
        "劲达电子", "通用企业", "新疆科华和昶生物科技", "亚太链接", "桂林阿尔法橡塑科技",
        "杭州富阳科拓机械", "瑞文国际贸易", "深圳卡斯普科技", "联盟电子科技", "阿尔法贸易投资",
        "上海亚联国际物流", "中印半导体", "深圳市查爱斯电子科技", "成都晶鑫微波技术", "中国盛世国际贸易",
        "易晶科技", "格莱特电子科技", "环球贝壳报关行", "华源世通科技", "景福线路板",
        "纽沃泰国际贸易", "诺普迅电子科技", "安士达电子", "PT科技亚洲", "融博通半导体",
        "上海英众信息科技", "深圳壹世达国际物流", "诗瓦贝光电", "森拓半导体", "泉州市南安特易通电子",
        "优创电路板", "永祺国际", "泽源科技", "北京壁仞科技开发", "广州壁仞集成电路",
        "杭州壁仞科技开发", "光线云", "摩尔线程智能科技", "上海壁仞", "超燃半导体",
        "上海新之砾企业发展", "珠海壁仞集成电路", "普莱德科技"
    ]

    entities_2024 = [
        "深圳斯彼得工业材料", "適合科技", "联合电子", "成都北展电子", "北京安怀信科技", "江西新拓实业", "联众集群科技",
        "深圳嘉思博科技", "思腾合力科技", "西安丽科创新信息技术", "AEE深圳一电航空技术", "北京北斗星通导航技术",
        "北京雷科防务技术", "北京瑞达康科技", "北京天海达科技", "北京中商鼎盛机电设备", "中电科芯片技术",
        "中电科思仪科技", "成都华日通讯技术", "成都纵横自动化技术", "中国电子科技集团公司电子装备", "中船重工鹏力",
        "中科星图", "和芯星通科技", "本源量子计算科技", "深圳一电科技", "苏州电讯电机厂",
        "太原依福特设备制造", "联合微电子中心", "西安恒达微波技术开发", "中科星图空间技术"
    ]

    entities_by_year = {
        2018: entities_2018,
        2019: entities_2019,
        2020: entities_2020,
        2021: entities_2021,
        2022: entities_2022,
        2023: entities_2023,
        2024: entities_2024
    }

    
    def find_entity_elem(entry):
        for yr in entities_by_year:
            for elem in entities_by_year[yr]:
                if elem in entry:
                    return yr
        return False

    result = {}
    for idx in range(len(L)):
        yr_found = find_entity_elem(L[idx])
        if yr_found:
            result[idx] = yr_found
    
    if result:
        return result
    return False


In [ ]:
list_with_firms['Entity_list'] = list_with_firms['author_add_cleaned'].parallel_apply(locate_entity)
list_with_firms

In [ ]:
list_with_firms[list_with_firms['Entity_list'] != False]

## Add variables

### Add var: firm colla with uni

In [ ]:
def contain_firm_uni(L):
    if 'u' in L:
        return True
    return False

list_with_firms = list_with_firms.copy()
list_with_firms.loc[:, 'firm_uni_colla'] = list_with_firms['category'].apply(contain_firm_uni)
list_with_firms

### Firm-paper level

In [ ]:
result_firms = {'paperid': [], 'firm_name': [], 'firm_uni_colla': [], 'entity_list': []}
for idx in range(list_with_firms.shape[0]):
    curr_entry = list_with_firms.iloc[idx]
    for firm_idx in curr_entry.contain_firms:
        result_firms['firm_name'].append(curr_entry.author_add_cleaned[firm_idx])
        result_firms['paperid'].append(curr_entry.paperid)
        result_firms['firm_uni_colla'].append(curr_entry.firm_uni_colla)
        result_firms['entity_list'].append(-1 if curr_entry['Entity_list'] == False else (-1 if firm_idx not in curr_entry['Entity_list'] else curr_entry['Entity_list'][firm_idx]))

df_result_firms = pd.DataFrame(result_firms)
df_result_firms

In [ ]:
df_result_firms = df_result_firms.drop_duplicates()
df_result_firms

### Add var: paper year

In [ ]:
paper_year = pd.read_csv(dataset_config['path_cnki'] + 'paper_list.txt', encoding='utf-8', delimiter='|', usecols=['paperid', 'pubtime', 'clc', 'journal', 'issn'], engine='pyarrow').dropna(subset=['pubtime'])
paper_year['year'] = paper_year['pubtime'].str[:4]
paper_year

In [ ]:
merge_year = pd.merge(df_result_firms, paper_year[['paperid', 'year', 'clc', 'journal', 'issn']], on='paperid')
merge_year

### Add var: funding information

In [ ]:
paper_funding = pd.read_csv(dataset_config['path_cnki'] + 'fund_list.txt', encoding='utf-8', delimiter='|')
paper_funding

In [ ]:
paper_funding['funding'] = paper_funding['fund_text'].apply(lambda x: 0 if pd.isna(x) or x == '' else 1)
paper_funding

In [ ]:
merge_year_funding = pd.merge(merge_year, paper_funding[['paperid', 'funding']], on='paperid')
merge_year_funding

In [ ]:
merge_year_funding[merge_year_funding.entity_list != -1]

### Consider overlap paper in the WOS (do not exclude here, we exclude these journal in the WOS)

In [ ]:
overlap_df = pd.read_csv(dataset_config['path_processed'] + "CNKI/overlap_cnkiwos_issn.csv")
filtered_df = merge_year_funding[~merge_year_funding['issn'].isin(overlap_df['issn_CNKI'])]
filtered_df = filtered_df.reset_index(drop=True)
filtered_df

## Export

In [ ]:
merge_year_funding.to_csv(dataset_config['path_processed'] + 'CNKI/02_CNKI_firm_paperid.csv', index=False, encoding='utf-8')

## Check: not identified

In [ ]:
def contains_o(L):
    if 'o' in L:
        return [idx for idx in range(len(L)) if L[idx] == 'o']
    else:
        return False
    
author_list['check'] = author_list['category'].apply(contains_o)
author_list

In [ ]:
df_check = author_list[author_list['check'] != False][['author_add', 'author_add_cleaned', 'check']]
df_check

In [ ]:
# Function to get corresponding author_add entries based on check indices
def get_author_add_by_check(df):
    result = []

    # Iterate over each row in the DataFrame
    for _, row in df.iterrows():
        author_add = row['author_add_cleaned']
        check_indices = row['check']

        # Select corresponding author_add entries based on the indices in check
        selected_adds = [author_add[i] for i in check_indices if i < len(author_add)]  # Safe index check
        result.extend(selected_adds)  # Add all selected items to the result list
        result.append(row['author_add'])
        result.append('-' * 20)

    return result

# Get the selected author_add items
selected_author_adds = get_author_add_by_check(df_check)
selected_author_adds

In [ ]:
not_identified = pd.DataFrame(selected_author_adds, columns=['name'])
not_identified.to_csv(dataset_config['path_processed'] + 'CNKI/02_CNKI_not_identified.csv', index=False, encoding='utf-8')

In [ ]:
df_check

### Calculate recognition rate

In [ ]:
recognition_rate = (1 - df_check.shape[0] / author_list.shape[0]) * 100
print(f"recognition_rate = {recognition_rate:.2f}%")